In [1]:
import csv
import json

def parse_row(row):
    """
    Attempts to cast specific dictionary values to their target data types.
    Returns a new dictionary with the converted types if successful.
    """
    converted = row.copy() # Copy to avoid partially altering a row that ultimately fails

    # These conversions will raise a ValueError if the value is blank or non-numeric
    converted["age"] = int(row["age"])
    converted["distance_km"] = float(row["distance_km"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])

    return converted

accepted = []
rejected = []
row_count = 0

# The brief requires newline="" for standard CSV reading
with open("data/appointments.csv", mode="r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)

    # Print the header exactly as required
    print("Header:", reader.fieldnames)

    for row in reader:
        row_count += 1
        try:
            # If parse_row succeeds, it appends to accepted
            clean_row = parse_row(row)
            accepted.append(clean_row)
        except ValueError:
            # If any int() or float() conversion fails, it jumps here
            rejected.append(row)

# Print the final counts to verify against the acceptance criteria
print(f"Total rows read: {row_count}")
print(f"Accepted rows: {len(accepted)}")
print(f"Rejected rows: {len(rejected)}")

# Dump the rejected rows to a JSON file with indentation
with open("rejected_rows.json", "w", encoding="utf-8") as json_file:
    json.dump(rejected, json_file, indent=2)

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows read: 1020
Accepted rows: 988
Rejected rows: 32


In [2]:
import pandas as pd

# Start fresh from the original CSV file
df = pd.read_csv("data/appointments.csv")

# Initial inspection
print("Initial shape:", df.shape)
print("\n--- Data Info ---")
df.info()
print("\n--- Missing Values ---")
print(df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("Raw distinct booking channels:", df["booking_channel"].nunique())

# (a) Drop exact duplicate rows
df = df.drop_duplicates().copy()
print(f"\nRow count after (a) dropping duplicates: {len(df)}")

# (b) Clean booking_channel spacing and casing
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print(f"Row count after (b) cleaning booking_channel: {len(df)}")
print("Cleaned booking channels:", df["booking_channel"].unique())

# (c) Convert date columns to datetime objects
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print(f"Row count after (c) converting dates: {len(df)}")

# (d) Drop rows where appt_date is missing/unreadable or status is missing
df = df.dropna(subset=["appt_date", "status"]).copy()
print(f"Row count after (d) dropping missing appt_date/status: {len(df)}")

Initial shape: (1020, 14)

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

--- Missing Values ---
appointment_id      0
p

In [3]:
import numpy as np

# (a) Treat any age below 1 or above 100 as invalid and set to missing
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
print(f"Invalid ages found: {invalid_age_mask.sum()}")
df.loc[invalid_age_mask, "age"] = np.nan
print(f"Total missing ages before imputation: {df['age'].isnull().sum()}")

# (b) Fill missing age and distance_km with their medians
age_median = df["age"].median()
dist_median = df["distance_km"].median()
print(f"Median age: {age_median}, Median distance: {dist_median}")

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)

# (c) Cap outliers in distance_km and fee_inr using IQR and np.clip
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    # Count values outside the fences before clipping
    capped_count = ((df[col] < lower_fence) | (df[col] > upper_fence)).sum()

    df[col] = np.clip(df[col], lower_fence, upper_fence)
    print(f"{col} -> Q1: {q1}, Q3: {q3}, Upper fence: {upper_fence}, Values capped: {capped_count}")

# (d) Round distance_km to 1 decimal and fee_inr to whole rupees
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("Largest fee after capping and rounding:", df["fee_inr"].max())

Invalid ages found: 5
Total missing ages before imputation: 17
Median age: 48.0, Median distance: 8.25
distance_km -> Q1: 4.625, Q3: 13.0, Upper fence: 25.5625, Values capped: 26
fee_inr -> Q1: 500.0, Q3: 787.5, Upper fence: 1218.75, Values capped: 3
Largest fee after capping and rounding: 1219


In [4]:
# Create new columns
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = df["status"].apply(lambda x: 1 if x == "No-Show" else 0)

print(f"lead_days runs from {df['lead_days'].min()} to {df['lead_days'].max()}")
print(f"Total no-shows: {df['no_show'].sum()} out of {len(df)} ({df['no_show'].mean()*100:.1f}%)")

# Extract the requested columns into a NumPy array
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Mean of each column
# axis=0 computes the mean down each column
means = np.mean(X, axis=0)
print(f"\nColumn means [age, distance_km, fee_inr, lead_days]:\n{means.round(2)}")

# (b) Count and share of rows with lead_days > 14
# lead_days is the 4th column (index 3)
lead_days_col = X[:, 3]
mask_lead_days = lead_days_col > 14

count_lead_14 = mask_lead_days.sum()
share_lead_14 = mask_lead_days.mean()
print(f"\nRows with lead_days > 14: {count_lead_14} ({share_lead_14*100:.1f}%)")

# (c) z-scores of distance_km and count of rows with |z| > 2
# distance_km is the 2nd column (index 1)
distance_col = X[:, 1]
# Standardize: (x - mean) / std (np.std defaults to population standard deviation)
z_scores = (distance_col - np.mean(distance_col)) / np.std(distance_col)

mask_z_scores = np.abs(z_scores) > 2
count_outliers = mask_z_scores.sum()
print(f"\nRows with |z| > 2 for distance_km: {count_outliers}")

lead_days runs from 1 to 45
Total no-shows: 218 out of 990 (22.0%)

Column means [age, distance_km, fee_inr, lead_days]:
[ 48.17   9.5  635.71  11.82]

Rows with lead_days > 14: 296 (29.9%)

Rows with |z| > 2 for distance_km: 53


In [5]:
import requests

clinics_df = pd.read_csv("data/clinics.csv")
temps = []
url = "https://api.open-meteo.com/v1/forecast"

for _, row in clinics_df.iterrows():
    clinic_name = row["name"] if "name" in clinics_df.columns else row["clinic_name"]
    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp_c = None
    fail_reason = ""

    # Attempt initial request + 1 retry on failure
    for attempt in range(2):
        try:
            resp = requests.get(url, params=params, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                hourly_temps = data.get("hourly", {}).get("temperature_2m")
                if hourly_temps and len(hourly_temps) > 0:
                    temp_c = hourly_temps[0]
                    break
                else:
                    fail_reason = "Missing 'hourly' or 'temperature_2m' in response"
            else:
                fail_reason = f"HTTP status {resp.status_code}"
        except requests.exceptions.RequestException as e:
            fail_reason = f"RequestException: {e}"

    if temp_c is None:
        print(f"Failed for {clinic_name}: {fail_reason}")

    temps.append(temp_c)

clinics_df["temp_c"] = temps
clinics_df.to_csv("clinics_enriched.csv", index=False)
print(clinics_df)

  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    17.4  
1    18.7  
2    19.9  
3    19.7  
4    19.7  
5    25.7  
6    18.3  
7    18.7  


In [6]:
# 1. Sort by appointment_id and format dates as YYYY-MM-DD strings
df = df.sort_values("appointment_id").reset_index(drop=True)
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

# 2. Enforce exact column order
cols_16 = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]
df_clean = df[cols_16].copy()

# 3. Verify uniqueness and zero missing values before exporting
print(f"Distinct appointment_ids: {df_clean['appointment_id'].nunique()} / {len(df_clean)}")
print(f"Total missing values across all 16 columns: {df_clean.isnull().sum().sum()}")

# 4. Save appointments_clean.csv
df_clean.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv with shape:", df_clean.shape)

# 5. Generate load_data.sql (clinics first, then appointments)
sql_statements = []

for _, r in clinics_df.iterrows():
    temp_val = "NULL" if pd.isna(r["temp_c"]) else f"{r['temp_c']}"
    stmt = (
        f"INSERT INTO clinics VALUES ("
        f"'{r['clinic_id']}', '{r['clinic_name']}', '{r['city']}', "
        f"'{r['tier']}', {r['latitude']}, {r['longitude']}, {temp_val});"
    )
    sql_statements.append(stmt)

for _, r in df_clean.iterrows():
    stmt = (
        f"INSERT INTO appointments VALUES ("
        f"'{r['appointment_id']}', '{r['patient_id']}', '{r['clinic_id']}', "
        f"'{r['booked_on']}', '{r['appt_date']}', {r['age']}, '{r['gender']}', "
        f"{r['distance_km']}, {r['fee_inr']}, '{r['booking_channel']}', "
        f"'{r['visit_type']}', '{r['reminder_sent']}', {r['prior_no_shows']}, "
        f"'{r['status']}', {r['lead_days']}, {r['no_show']});"
    )
    sql_statements.append(stmt)

with open("load_data.sql", "w", encoding="utf-8") as f:
    f.write("\n".join(sql_statements) + "\n")

print(f"Saved load_data.sql with {len(sql_statements)} INSERT statements.")

Distinct appointment_ids: 990 / 990
Total missing values across all 16 columns: 0
Saved appointments_clean.csv with shape: (990, 16)
Saved load_data.sql with 998 INSERT statements.
